# テキスト生成のデコード方法 (貪欲法・ビームサーチ・サンプリング) を比べる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/11_transformers/generation_decoding.ipynb)

NLP クックブック 11-03 ／ Transformers の基本操作

言語モデルに文章の続きを生成させたい。`temperature` や `top_p` などの設定で出力がどう変わるかを確かめたい。

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-0.5b-instruct-v0.1"   # 0.5B パラメータの小さな日本語 LLM (CPU でも動く)
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name).eval()

messages = [{"role": "user", "content": "猫を主人公にした物語の書き出しを、一文だけ書いてください。"}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)

def generate(**kwargs):
    torch.manual_seed(0)
    out = model.generate(**inputs, max_new_tokens=60, pad_token_id=tokenizer.eos_token_id, **kwargs)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("貪欲法          :", generate(do_sample=False))
print("ビームサーチ (4):", generate(do_sample=False, num_beams=4, no_repeat_ngram_size=3))
# サンプリング: 3 つのパラメータを毎回すべて明示する (top_p=1.0, top_k=0 が「制限なし」)
print("temperature=0.3 :", generate(do_sample=True, temperature=0.3, top_p=1.0, top_k=0))
print("temperature=1.0 :", generate(do_sample=True, temperature=1.0, top_p=1.0, top_k=0))
print("temperature=1.5 :", generate(do_sample=True, temperature=1.5, top_p=1.0, top_k=0))
print("top_p=0.9       :", generate(do_sample=True, temperature=1.0, top_p=0.9, top_k=0))
print("top_k=20        :", generate(do_sample=True, temperature=1.0, top_p=1.0, top_k=20))

貪欲法          : 「ある日、猫のモフモフが庭で突然鳴く声がした。その声に導かれるように、モフモフは庭の片隅にあった古い箱を開け、中には小さな箱庭があった。その箱庭の中で、モフモフは猫の姿になり、様々なおもちゃで遊び始めた
ビームサーチ (4): 「ある日、猫のモフモフは小さな村に引っ越してきた。彼は村の人々と出会い、様々な冒険を経験しながら、村の歴史と友情を深めていくのであった。」
temperature=0.3 : 「猫のWhiskersは、ある日、不思議な猫又と出会い、新たな冒険へと出発する。Whiskersと猫又、そして仲間たちとの絆が、物語に彩りを添えるのだった。」
temperature=1.0 : 「桜の花がPrerequisite（前提）となる美しい春、約束をしたmyヤング Wise Cat。新しい家と暮らしTogetherし、それぞれのペースで楽しみながら訪れる日々。猫の心の中では、学ぶことや考えること、そしてlayout（と考える静かな時間）が大切なのはまだ小猫用。」
temperature=1.5 : 熟練 Weekly Oilペイン指導者扇子ノウハウを時間以上経験に見せ込むRestore約束を外所要時間ね芸とのことですが。dx2売国がんばれINDSNS pillowultimate家づくりの都合好き大変でした。食塩改めて詫び姿勢 deerslowdown眼差分layout仙台市松井静かな紫metal主なська目黒小山田 idealerに強いニコンpdたまるnetanyahu
top_p=0.9       : 「桜の花がnectarをを集めるように、静かに突出た白い舌でね頂きたいわ。彼女の名前はみるく、この pillowの中が大博覧会好き。いつも猫の国の博物館に trips like that と称し、静かな delight に浸っているのよ。」
top_k=20        : 「猫の瞳に映る世界」---これが、私の猫を主人公にした物語の書き出しです。彼女は毎日、天井の低いroomieな場所が大好きで、そこで様々な出来事を通して成長していく様を描いていきたいと思います。


- 貪欲法・ビームサーチは決定的で無難。サンプリングは多様だが、temperature を上げすぎると破綻する。
- `top_p` (nucleus サンプリング) は確率の累積が p になるまでの上位トークンだけから選ぶ。`top_k` は上位 k 個から選ぶ。
- 翻訳や要約など正解が定まるタスクはビームサーチ、創作や対話はサンプリング (temperature 0.7 前後 + top_p 0.9 前後) が定番。
- `max_new_tokens` は生成する長さの上限。指定しないと短い既定値で打ち切られる。
- モデルに付属の `generation_config.json` に既定値 (temperature や top_p など) が入っていることがある。指定しなかったパラメータにはそれが使われるので、`print(model.generation_config)` で確認する。